In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv("Digital_Twin_Predictive_Maintenance_1500.csv")

print("Dataset Shape:", df.shape)

# =====================================================
# DROP UNUSED COLUMNS
# =====================================================

if "Timestamp" in df.columns:
    df.drop("Timestamp", axis=1, inplace=True)

if "Maintenance_Needed" in df.columns:
    df.drop("Maintenance_Needed", axis=1, inplace=True)

# =====================================================
# ENCODE TARGET
# =====================================================

label_encoder = LabelEncoder()

df["Fault_Type"] = label_encoder.fit_transform(
    df["Fault_Type"]
)

# Save encoder
joblib.dump(
    label_encoder,
    "fault_type_encoder.pkl"
)

# =====================================================
# FEATURES / TARGET
# =====================================================

X = df.drop("Fault_Type", axis=1)

y = df["Fault_Type"]

print("\nFeatures Used:")
print(X.columns.tolist())

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain Size:", len(X_train))
print("Test Size :", len(X_test))

# =====================================================
# XGBOOST MODEL
# =====================================================

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=5,
    random_state=42,
    eval_metric="mlogloss"
)

# =====================================================
# TRAIN
# =====================================================

model.fit(X_train, y_train)

# =====================================================
# PREDICTION
# =====================================================

y_pred = model.predict(X_test)

# =====================================================
# ACCURACY
# =====================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n==============================")
print("MODEL PERFORMANCE")
print("==============================")

print(f"Accuracy : {accuracy*100:.2f}%")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

# =====================================================
# CONFUSION MATRIX
# =====================================================

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:\n")
print(cm)

# =====================================================
# FEATURE IMPORTANCE
# =====================================================

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\n==============================")
print("FEATURE IMPORTANCE")
print("==============================")

print(importance_df)

# =====================================================
# SAVE MODEL
# =====================================================

joblib.dump(
    model,
    "digital_twin_xgboost_model.pkl"
)

print("\nModel Saved Successfully")

print("Saved Files:")
print("1. digital_twin_xgboost_model.pkl")
print("2. fault_type_encoder.pkl")

Dataset Shape: (1500, 12)

Features Used:
['Unnamed: 0', 'RPM_Commanded', 'RPM_Actual', 'Current', 'Expected_Current', 'Current_Deviation_Percent', 'RPM_Drop_Percent', 'Efficiency_Index', 'Estimated_Temperature', 'Health_Score']

Train Size: 1200
Test Size : 300

MODEL PERFORMANCE
Accuracy : 99.00%

Classification Report:

                  precision    recall  f1-score   support

    Bearing_Wear       1.00      1.00      1.00        60
        Critical       1.00      1.00      1.00        60
          Normal       1.00      1.00      1.00        60
Overheating_Risk       0.97      0.98      0.98        60
      Overloaded       0.98      0.97      0.97        60

        accuracy                           0.99       300
       macro avg       0.99      0.99      0.99       300
    weighted avg       0.99      0.99      0.99       300


Confusion Matrix:

[[60  0  0  0  0]
 [ 0 60  0  0  0]
 [ 0  0 60  0  0]
 [ 0  0  0 59  1]
 [ 0  0  0  2 58]]

FEATURE IMPORTANCE
                   